# Telemetry Bronze to Silver Pipeline

Batch pipeline that reads power meter events from the bronze table, uses the `analytical_reference_date` table to track per-device watermarks (with a 24-hour fallback for first-time devices), transforms the nested `readings` array into a narrow silver schema (device_id, measurement_name, measurement_value, measurement_time, kafka_timestamp), and updates the reference dates after each run.

In [0]:
%pip install /Workspace/Users/imbipin8858@gmail.com/user-activity-analytics/dist/analytical_pipeline-0.0.1-py3-none-any.whl

In [0]:
dbutils.library.restartPython()

In [0]:
bronze_table_name='databricks_learning.custom.power_meter_simulator_events_bronze'
reference_table_name='databricks_learning.custom.analytical_reference_date'
silver_table_name='databricks_learning.custom.power_meter_simulator_events_silver'
audit_table_name='databricks_learning.custom.pipeline_audit'

# bronze_table_name=''
# reference_table_name=''
# silver_table_name=''

dbutils.widgets.text(name='bronze_table_name',defaultValue=bronze_table_name, label='bronze_table_name')
dbutils.widgets.text(name='reference_table_name',defaultValue=reference_table_name, label='reference_table_name')
dbutils.widgets.text(name='silver_table_name',defaultValue=silver_table_name, label='silver_table_name')
dbutils.widgets.text(name='audit_table_name',defaultValue=audit_table_name, label='audit_table_name')

bronze_table_name = dbutils.widgets.get('bronze_table_name')
reference_table_name = dbutils.widgets.get('reference_table_name')
silver_table_name = dbutils.widgets.get('silver_table_name')
audit_table_name = dbutils.widgets.get('audit_table_name')


In [0]:
from analytical_pipeline.utils.logger import get_logger
from analytical_pipeline.utils.audit import AuditLogger
import uuid

logger = get_logger('DEV:bronze-to-silver')
logger.info('Logger initialized for bronze-to-silver pipeline')

run_id = str(uuid.uuid4())
audit_logger = AuditLogger(audit_table_name)
audit_logger.start_run(run_id, 'telemetry-bronze-to-silver', 'bronze-to-silver')
logger.info(f'Audit run started: {run_id}')
logger.info(f'Audit table: {audit_table_name}')

In [0]:
logger.info(f'Reading reference table: {reference_table_name}')
logger.info(f'Reading bronze table: {bronze_table_name}')
reference_date_df = spark.read.format('delta').table(reference_table_name)
bronze_measurements_df = spark.read.format('delta').table(bronze_table_name)
logger.info('Tables loaded successfully')

In [0]:
from pyspark.sql.functions import col, current_timestamp, expr, lit, when, max as spark_max

# Read the reference date table
reference_date_df = spark.read.format('delta').table(reference_table_name)

# Get distinct device_ids from bronze table
distinct_devices_df = bronze_measurements_df.select("device_id").distinct()

# Build watermark lookup: for each device, use last_processed_ts if available, else fall back to 24 hours ago
watermark_df = distinct_devices_df.join(
    reference_date_df.select("device_id", "last_processed_ts"),
    on="device_id",
    how="left"
).withColumn(
    "watermark_ts",
    when(col("last_processed_ts").isNotNull(), col("last_processed_ts"))
    .otherwise(current_timestamp() - expr("INTERVAL 24 HOURS"))
)

display(watermark_df)
logger.info('Watermark lookup built successfully')

In [0]:
# Filter bronze data: only keep rows where kafka_timestamp is newer than the device's watermark
bronze_filtered = bronze_measurements_df.join(
    watermark_df.select("device_id", "watermark_ts"),
    on="device_id",
    how="inner"
).filter(col("kafka_timestamp") > col("watermark_ts"))

bronze_row_count = bronze_filtered.count()
logger.info(f"Filtered bronze rows to process: {bronze_row_count}")
if bronze_row_count == 0:
    logger.warning("No new data to process. Exiting.")
    audit_logger.mark_success(run_id, 'bronze-to-silver', 0)
    dbutils.notebook.exit("No new data to process")
else:
    display(bronze_filtered.select("device_id", "generated_at", "kafka_timestamp").limit(10))

In [0]:
from pyspark.sql.functions import explode, array, struct, lit

# Explode the readings array so each reading struct becomes a row
exploded_df = bronze_filtered.select(
    col("device_id"),
    col("kafka_timestamp"),
    explode(col("readings")).alias("reading")
)

# Measurement field names from the readings struct
measurement_fields = [
    "voltage_r", "voltage_y", "voltage_b",
    "current_r", "current_y", "current_b",
    "active_power_kw", "reactive_power_kvar", "apparent_power_kva",
    "power_factor", "frequency_hz", "energy_kwh",
    "thd_voltage", "thd_current", "temperature_c"
]

# Build an array of struct(measurement_name, measurement_value) and explode to unpivot
silver_df = exploded_df.select(
    col("device_id"),
    col("kafka_timestamp"),
    col("reading.measurement_time").cast("timestamp").alias("measurement_time"),
    explode(array([
        struct(lit(name).alias("measurement_name"), col(f"reading.{name}").cast("double").alias("measurement_value"))
        for name in measurement_fields
    ])).alias("measurement")
).select(
    col("device_id"),
    col("measurement.measurement_name").alias("measurement_name"),
    col("measurement.measurement_value").alias("measurement_value"),
    col("measurement_time"),
    col("kafka_timestamp")
)

silver_row_count = silver_df.count()
logger.info(f"Total silver rows to write: {silver_row_count}")
display(silver_df.limit(20))

In [0]:
# Register the transformed DataFrame as a temp view for MERGE
silver_df.createOrReplaceTempView("silver_updates")

logger.info(f'Merging {silver_row_count} rows into silver table: {silver_table_name}')

spark.sql(f"""
MERGE INTO {silver_table_name} AS target
USING silver_updates AS source
ON  target.device_id = source.device_id
    AND target.measurement_name = source.measurement_name
    AND target.measurement_time = source.measurement_time
    AND target.kafka_timestamp = source.kafka_timestamp
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

logger.info("MERGE into silver table completed.")

In [0]:
# Compute the latest kafka_timestamp per device from the newly processed data
logger.info('Updating reference dates with latest watermarks per device')
new_watermarks = silver_df.groupBy("device_id").agg(
    spark_max("kafka_timestamp").alias("last_processed_ts")
).withColumn("updated_at", current_timestamp())

new_watermarks.createOrReplaceTempView("new_watermarks")

# Upsert into the reference date table
spark.sql(f"""
MERGE INTO {reference_table_name} AS target
USING new_watermarks AS source
ON target.device_id = source.device_id
WHEN MATCHED THEN UPDATE SET
    target.last_processed_ts = source.last_processed_ts,
    target.updated_at = source.updated_at
WHEN NOT MATCHED THEN INSERT *
""")

logger.info("Reference dates updated.")
display(spark.read.table(reference_table_name))

In [0]:
audit_logger.mark_success(run_id, 'bronze-to-silver', silver_row_count)
logger.info(f'Pipeline completed successfully! Run ID: {run_id}, Records processed: {silver_row_count}')

## Pipeline Summary

1. **Watermark lookup**: For each device in bronze, retrieves `last_processed_ts` from the reference table. Falls back to 24 hours ago for devices with no reference date.
2. **Filter**: Keeps only bronze rows where `kafka_timestamp` > device watermark.
3. **Transform**: Explodes the `readings` array and unpivots 14 measurement fields into rows (`device_id`, `measurement_name`, `measurement_value`, `measurement_time`, `kafka_timestamp`).
4. **Merge**: Idempotently upserts into the silver table using a composite key.
5. **Reference update**: Updates `last_processed_ts` per device with the latest `kafka_timestamp` processed.
6. **Audit**: Logs `start_run` at pipeline start and `mark_success` with `records_processed` (silver row count) at the end. On failure, the audit row remains in `PROCESSING` status for monitoring.